# H4: Foraging Profiles and Optimality (Bayesian)

All regressions use bambi Bayesian linear models with 95% HDI inference.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, os
import bambi as bmb, arviz as az
from scipy.stats import pearsonr, zscore
from scipy.special import expit
from config import *

%matplotlib inline

# Load master + MCMC params
master = pd.read_csv(MODEL_DIR / 'master_subject_df.csv', index_col='subj')
mp = pd.read_csv(MODEL_DIR / 'mcmc_m4_params.csv').set_index('subj')
shared = sorted(set(master.index) & set(mp.index))
master.loc[shared,'omega'] = mp.loc[shared,'omega']
master.loc[shared,'kappa'] = mp.loc[shared,'kappa']
master['log_omega'] = np.log(master['omega']); master['log_kappa'] = np.log(master['kappa'])
master['omega_z'] = zscore(master['log_omega'].values)
master['kappa_z'] = zscore(master['log_kappa'].values)

# Behavioral data
beh = pd.read_csv(DATA_DIR/'behavior_rich.csv', low_memory=False)
beh = beh[~beh['subj'].isin(EXCLUDE)]
cdf = beh[beh['type']==1].copy(); cdf['T_round'] = cdf['threat'].round(1)
cells = pd.read_csv(VIGOR_DIR/'cell_means.csv')
cells = cells[~cells['subj'].isin(EXCLUDE)]
attack = beh[beh['isAttackTrial']==1]
master['escape_rate'] = attack.groupby('subj').apply(lambda x:(x['trialEndState']=='escaped').mean())
master['earnings'] = beh.groupby('subj')['trialReward'].sum()
master['p_heavy'] = cdf.groupby('subj')['choice'].mean()
master['mean_vigor'] = cells.groupby('subj')['mean_rate'].mean()

# Optimality
opt_map = {}
for (T,D),grp in cdf.groupby(['T_round','distance_H']):
    er_h = grp[grp['choice']==1]['trialReward'].mean() if (grp['choice']==1).sum()>0 else -99
    er_l = grp[grp['choice']==0]['trialReward'].mean() if (grp['choice']==0).sum()>0 else -99
    opt_map[(T,D)] = 1 if er_h > er_l else 0
cdf['optimal'] = cdf.apply(lambda r:opt_map.get((r['T_round'],r['distance_H']),np.nan),axis=1)
cdf['is_opt'] = (cdf['choice']==cdf['optimal']).astype(int)
cdf['err_type'] = np.where(cdf['is_opt']==1,'correct',np.where(cdf['choice']==0,'overcautious','reckless'))
so = cdf.groupby('subj').agg(pct_opt=('is_opt','mean'),n_oc=('err_type',lambda x:(x=='overcautious').sum()),
    n_rk=('err_type',lambda x:(x=='reckless').sum()),n_err=('is_opt',lambda x:(x==0).sum()))
so['oc_ratio'] = so['n_oc']/so['n_err'].clip(1)
for c in so.columns:
    if c in master.columns: master = master.drop(columns=[c])
master = master.join(so, how='left')
master['angle_z'] = zscore(np.arctan2(master['kappa_z'],master['omega_z']).values)

print(f'Master: {len(master)} subjects (MCMC params)')


Master: 290 subjects (MCMC params)


## H4a: Avoidance sensitivity predicts escape

In [2]:
df = master[['escape_rate','omega_z','kappa_z']].dropna()
m = bmb.Model('escape_rate ~ omega_z + kappa_z', data=df)
r = m.fit(**BKW)
s = az.summary(r, hdi_prob=0.95)
print('H4a — escape ~ omega + kappa (Bayesian)')
print(s[['mean','hdi_2.5%','hdi_97.5%']].to_string())
lo_a = s.loc['omega_z','hdi_2.5%']
p_a = lo_a > 0
print(f'\nH4a: {"PASS ✓" if p_a else "FAIL ✗"} (omega 95% HDI excludes zero, positive)')

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, omega_z, kappa_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.


H4a — escape ~ omega + kappa (Bayesian)
            mean  hdi_2.5%  hdi_97.5%
sigma      0.240     0.221      0.260
Intercept  0.384     0.356      0.410
omega_z    0.060     0.029      0.093
kappa_z   -0.003    -0.033      0.029

H4a: PASS ✓ (omega 95% HDI excludes zero, positive)


## H4b: Overcaution + avoidance sensitivity

In [3]:
oc_pct = master['oc_ratio'].mean()*100
pass_pct = oc_pct > 65
df = master[['oc_ratio','omega_z']].dropna()
m = bmb.Model('oc_ratio ~ omega_z', data=df)
r = m.fit(**BKW)
s = az.summary(r, hdi_prob=0.95)
lo_b = s.loc['omega_z','hdi_2.5%']
p_b = lo_b > 0
print(f'H4b — Overcaution: {oc_pct:.0f}% {"✓" if pass_pct else "✗"}')
print(f'  omega → OC: β={s.loc["omega_z","mean"]:+.4f}, 95% HDI=[{lo_b:+.4f},{s.loc["omega_z","hdi_97.5%"]:+.4f}]')
print(f'  H4b: {"PASS ✓" if p_b and pass_pct else "FAIL ✗"}')

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, omega_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.


H4b — Overcaution: 79% ✓
  omega → OC: β=+0.1770, 95% HDI=[+0.1630,+0.1930]
  H4b: PASS ✓


## H4c: Effort cost predicts pressing intensity

In [4]:
df = master[['mean_vigor','kappa_z']].dropna()
m = bmb.Model('mean_vigor ~ kappa_z', data=df)
r = m.fit(**BKW)
s = az.summary(r, hdi_prob=0.95)
hi_c = s.loc['kappa_z','hdi_97.5%']
p_c = hi_c < 0
print(f'H4c — kappa → vigor: β={s.loc["kappa_z","mean"]:+.4f}, 95% HDI=[{s.loc["kappa_z","hdi_2.5%"]:+.4f},{hi_c:+.4f}]')
print(f'H4c: {"PASS ✓" if p_c else "FAIL ✗"} (95% HDI excludes zero, negative)')

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, kappa_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.


H4c — kappa → vigor: β=-0.1940, 95% HDI=[-0.2150,-0.1730]
H4c: PASS ✓ (95% HDI excludes zero, negative)


## H4d: Effort-driven avoidance less optimal

In [5]:
df = master[['pct_opt','angle_z']].dropna()
m = bmb.Model('pct_opt ~ angle_z', data=df)
r = m.fit(**BKW)
s = az.summary(r, hdi_prob=0.95)
hi_d = s.loc['angle_z','hdi_97.5%']
p_d = hi_d < 0
print(f'H4d — angle → optimality: β={s.loc["angle_z","mean"]:+.4f}, 95% HDI=[{s.loc["angle_z","hdi_2.5%"]:+.4f},{hi_d:+.4f}]')
print(f'H4d: {"PASS ✓" if p_d else "FAIL ✗"} (95% HDI excludes zero, negative)')

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, angle_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 1 seconds.


H4d — angle → optimality: β=-0.0410, 95% HDI=[-0.0550,-0.0260]
H4d: PASS ✓ (95% HDI excludes zero, negative)


## H4e: W(u) consistency predicts earnings

In [6]:
gamma=0.76; hazard=0.481; sp=0.25; C_pen=5.0
ug_np = np.linspace(0.1,1.5,100)
def uV(om,kap,T,D,R,req):
    speed=expit((ug_np-0.25*req)/sp)
    S=np.exp(-hazard*T**gamma*D/np.clip(speed,.01,None))
    W=S*R-(1-S)*om*(R+C_pen)-kap*(ug_np-req)**2*D
    return ug_np[W.argmax()], W.max()-kap*req*D

ch_cons={}; int_dev={}
for s in master.index:
    om=master.loc[s,'omega']; kap=master.loc[s,'kappa']
    st=cdf[cdf['subj']==s]; c=0; t=0
    for _,tr in st.iterrows():
        _,VH=uV(om,kap,tr['threat'],tr['distance_H'],5.,.9)
        _,VL=uV(om,kap,tr['threat'],1.,1.,.4)
        if (1 if VH>VL else 0)==int(tr['choice']): c+=1
        t+=1
    ch_cons[s]=c/t if t>0 else np.nan
    sc=cells[cells['subj']==s]
    if len(sc)<5: int_dev[s]=np.nan; continue
    ps=[]; ac=[]
    for _,cl in sc.iterrows():
        R=5. if cl['is_heavy']==1 else 1.; req=.9 if cl['is_heavy']==1 else .4
        u,_=uV(om,kap,cl['T_round'],cl['actual_dist'],R,req)
        ps.append(u); ac.append(cl['mean_rate'])
    int_dev[s]=np.sqrt(np.mean((np.array(ps) - np.array(ac))**2))

cc_s = pd.Series(ch_cons).reindex(master.index)
id_s = pd.Series(int_dev).reindex(master.index)
# Drop NaN before zscore
valid = cc_s.notna() & id_s.notna() & master['earnings'].notna()
master.loc[valid,'cc_z'] = zscore(cc_s[valid].values)
master.loc[valid,'id_z'] = zscore(id_s[valid].values)
df=master[['earnings','cc_z','id_z']].dropna()
m=bmb.Model('earnings ~ cc_z + id_z', data=df)
r=m.fit(**BKW)
s=az.summary(r, hdi_prob=0.95)
print('H4e — earnings ~ choice_cons + intensity_deviation (Bayesian)')
print(s[['mean','hdi_2.5%','hdi_97.5%']].to_string())
p_ch=s.loc['cc_z','hdi_2.5%']>0; p_vi=s.loc['id_z','hdi_97.5%']<0
print(f'\nChoice (β>0): {"PASS ✓" if p_ch else "FAIL ✗"}, Intensity dev (β<0): {"PASS ✓" if p_vi else "FAIL ✗"}')
print(f'H4e: {"PASS ✓" if p_ch and p_vi else "FAIL ✗"}')

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, cc_z, id_z]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 2 seconds.


H4e — earnings ~ choice_cons + intensity_deviation (Bayesian)
             mean  hdi_2.5%  hdi_97.5%
sigma      84.320    77.639     91.281
Intercept   6.464    -3.751     15.419
cc_z       14.229     4.066     23.957
id_z      -19.316   -28.764     -9.446

Choice (β>0): PASS ✓, Intensity dev (β<0): PASS ✓
H4e: PASS ✓


## Summary

In [7]:
print('H4 SUMMARY (Bayesian) — all tests use 95% HDI excludes zero')
print('='*60)
results = [('H4a','ω → escape',p_a),('H4b','OC %',pass_pct),('H4b','ω → OC',p_b),
           ('H4c','κ → vigor',p_c),('H4d','angle → opt',p_d),
           ('H4e','choice → earn',p_ch),('H4e','int_dev → earn',p_vi)]
for h,t,p in results: print(f'  {h:<6} {t:<20} {"✓" if p else "✗"}')
n=sum(1 for *_,p in results if p)
print(f'\n{n}/{len(results)} tests pass.')

H4 SUMMARY (Bayesian) — all tests use 95% HDI excludes zero
  H4a    ω → escape           ✓
  H4b    OC %                 ✓
  H4b    ω → OC               ✓
  H4c    κ → vigor            ✓
  H4d    angle → opt          ✓
  H4e    choice → earn        ✓
  H4e    int_dev → earn       ✓

7/7 tests pass.
